In [ ]:
# Colab bootstrap: install requirements and stage the course data next to the notebook
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/griu/ai-ml-finance-hands-on"
    REPO_DIR = Path("/content/ai-ml-finance-hands-on")
    NOTEBOOK_DIR = Path.cwd()
    DATA_TARGET = NOTEBOOK_DIR / "data"
    REQUIREMENTS_FILE = REPO_DIR / "requirements-colab.txt"

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)],
        check=True,
    )

    shutil.copytree(REPO_DIR / "data", DATA_TARGET, dirs_exist_ok=True)
    print(f"Colab environment detected. Data copied to: {DATA_TARGET.resolve()}")
else:
    print("Local environment detected. Colab bootstrap skipped.")


# 05d — Supervised Neural Networks with PyTorch

**Course:** Data Analysis, AI and Machine Learning in Finance — Ferran Carrascosa  
**Topic 2.3.1.d — Neural networks within supervised ML**

---

### What this compact notebook covers

| Section | Topic |
|---------|-------|
| 0 | Why include a neural-network example |
| 1 | Build a compact numeric table |
| 2 | Logistic baseline on the same data |
| 3 | Small PyTorch MLP |
| 4 | Compare results |
| 5 | Why TensorFlow/Keras is also worth knowing |

**Expected time:** 10–15 minutes  
**Role:** optional compact support notebook

## Section 0 — Why include a neural-network example

Neural networks are important in modern AI, but in many tabular finance problems they are not automatically better than strong tree-based methods. This notebook is intentionally compact: the goal is to demystify the workflow, not to build a deep-learning engineering course.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from pathlib import Path
_candidates = [Path('data/processed'), Path('../data/processed')]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Cannot locate data/processed/. Launch from project root or notebooks/.')

import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression


In [ ]:
df = pd.read_csv(DATA_DIR / 'dataset_F_deep_learning_awareness.csv').copy()
X = pd.get_dummies(df.drop(columns=['customer_id', 'attrition_flag']), drop_first=True)
y = df['attrition_flag'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype('float32')
X_test_sc = scaler.transform(X_test).astype('float32')
X_train_sc.shape

## Section 1 — Logistic baseline on the same split

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_sc, y_train)
prob_lr = logreg.predict_proba(X_test_sc)[:, 1]
pred_lr = logreg.predict(X_test_sc)

def metric_row(name, y_true, y_pred, y_prob):
    return {'model': name, 'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'auc': roc_auc_score(y_true, y_prob)}

res_lr = metric_row('Logistic baseline', y_test, pred_lr, prob_lr)
res_lr

## Section 2 — Small PyTorch MLP

In [ ]:
torch.manual_seed(42)
X_train_t = torch.tensor(X_train_sc)
y_train_t = torch.tensor(y_train.reshape(-1, 1)).float()
X_test_t = torch.tensor(X_test_sc)

model = nn.Sequential(nn.Linear(X_train_sc.shape[1], 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1))
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_history = []
for epoch in range(12):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))
loss_history[:3], loss_history[-3:]

In [ ]:
plt.plot(loss_history)
plt.title('Training loss — small PyTorch MLP')
plt.xlabel('Epoch')
plt.ylabel('BCE loss')
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    logits_test = model(X_test_t).numpy().ravel()
prob_nn = 1 / (1 + np.exp(-logits_test))
pred_nn = (prob_nn >= 0.5).astype(int)
res_nn = metric_row('PyTorch MLP', y_test, pred_nn, prob_nn)
res_nn

## Section 3 — Compare results

In [ ]:
comparison = pd.DataFrame([res_lr, res_nn]).round(3)
comparison

In [ ]:
comparison.set_index('model')[['auc', 'recall', 'f1']].plot(kind='bar')
plt.ylim(0, 1)
plt.title('Logistic baseline vs. small PyTorch MLP')
plt.xticks(rotation=10)
plt.show()

## Section 4 — Why TensorFlow / Keras is also worth knowing

In the wider ecosystem, **TensorFlow/Keras** is another major deep-learning framework. In this course we keep it at awareness level, while using **PyTorch** for a compact hands-on example.

### Diagnose → transform → validate → justify
- **Diagnose:** later AI topics assume basic familiarity with this family of models.
- **Transform:** categorical variables were encoded and all features were scaled.
- **Validate:** the MLP was compared directly with logistic regression.
- **Justify:** the goal is not to oversell deep learning, but to understand where it fits.

### Key ideas
1. PyTorch lets us show the neural-network workflow explicitly.
2. A small MLP is enough for conceptual grounding.
3. In tabular finance, neural networks are one option among several—not the default winner.